# CSA Speed Benchmark

Controls timing benchmarks for `CSADataHandler` and `JourneyPlanner`.

**Workflow**
1. Run **Setup** once per kernel session.
2. Run **Prepare** only when the planner has not been prepared yet, or when you intentionally want to rebuild/refetch data.
3. After editing `src/routing/journey_planner_v3.py`, run the **reload code only** cell. It keeps the prepared in-memory data and swaps in the latest planner code.
4. Run any benchmark cell - set `iterations` to reduce noise from fluctuations.

The instrumented code inside the source files prints a per-function breakdown automatically.
The bench functions here add aggregate stats (avg / min / max / stdev).

## Setup

In [ ]:
import importlib
import os
import sys

cwd = os.path.abspath(os.getcwd())
project_root = cwd if os.path.exists(os.path.join(cwd, "src")) else os.path.abspath(os.path.join(cwd, ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

import src.config.settings as _s
import src.data.csa_data_handler as _cdh
import src.routing.journey_planner_v3 as _jp
import tests.test_CSA as _bench


def _prepared_summary(p):
    if not getattr(p, "prepared", False):
        return " (planner is not prepared yet)"

    days = getattr(p, "connections_by_day", None) or {}
    n_day_connections = sum(len(v) for v in days.values())
    n_stops = len(getattr(p, "stops", None) or [])
    n_trips = getattr(p, "n_trips", 0)
    return f" ({n_stops:,} stops, {n_trips:,} trips, {n_day_connections:,} day-connections)"


def reload_planner_code(preserve_prepared=True):
    """Reload planner/benchmark code while keeping prepared CSA data in memory."""
    global settings, get_settings, JourneyPlanner
    global bench_prepare, bench_plan_candidates, bench_plan_profile, bench_route, bench_route_backward
    global _s, _cdh, _jp, _bench, planner

    old_planner = globals().get("planner")

    for module in (_s, _cdh, _jp, _bench):
        importlib.reload(module)

    get_settings = _s.get_settings
    JourneyPlanner = _jp.JourneyPlanner
    bench_prepare = _bench.bench_prepare
    bench_plan_candidates = _bench.bench_plan_candidates
    bench_route = _bench.bench_route
    bench_plan_profile = _bench.bench_plan_profile
    bench_route_backward = _bench.bench_route_backward
    try:
        settings = get_settings()
    except Exception:
        if old_planner is not None and hasattr(old_planner, "settings"):
            settings = old_planner.settings
        elif "settings" in globals():
            settings = globals()["settings"]
        else:
            raise

    new_planner = JourneyPlanner(settings=settings)

    if preserve_prepared and old_planner is not None and getattr(old_planner, "prepared", False):
        for name, value in vars(old_planner).items():
            setattr(new_planner, name, value)

        # Keep current settings/schema fresh, while preserving the materialized indexes.
        new_planner.settings = settings
        new_planner.schema = getattr(old_planner, "schema", new_planner.schema)
        new_planner.shared_schema = getattr(old_planner, "shared_schema", new_planner.shared_schema)

        if getattr(new_planner, "data_handler", None) is not None:
            try:
                new_planner.data_handler.__class__ = _cdh.CSADataHandler
            except TypeError:
                pass

        planner = new_planner
        print("Reloaded journey_planner_v3.py; preserved prepared data" + _prepared_summary(planner))
    else:
        planner = new_planner
        print("Reloaded journey_planner_v3.py; created a fresh unprepared planner")

    return planner


planner = reload_planner_code(preserve_prepared=True)

Reloaded journey_planner_v3.py; preserved prepared data (25,752 stops, 1,340,906 trips, 35,752,786 day-connections)


In [23]:
# Run this after editing src/routing/journey_planner_v3.py.
# This swaps in the latest method code while keeping prepared data in memory.
planner = reload_planner_code()

Reloaded journey_planner_v3.py; preserved prepared data (25,752 stops, 1,340,906 trips, 35,752,786 day-connections)


In [24]:
import src.config.settings

LAUSANNE_REGION_UUIDS = (
    "a7a21b73-6ffe-4fbf-a635-6e2b961f3072",
    "e168fd57-f57a-4075-a350-0dcfbb55147f",
)
REGION_UUIDS = settings.region_uuids or LAUSANNE_REGION_UUIDS

START_STOP = 8501120   # Lausanne
END_STOP   = 8501117   # Renens VD


print(f"Regions : {REGION_UUIDS}")
print(f"Stops   : {START_STOP} → {END_STOP}")

Regions : ('a7a21b73-6ffe-4fbf-a635-6e2b961f3072', 'e168fd57-f57a-4075-a350-0dcfbb55147f')
Stops   : 8501120 → 8501117


## Prepare

Run this only when `planner.prepared` is `False`, or when you intentionally want to rebuild/refetch data.

- `REBUILD_TABLES = True` -> drop and recreate Trino tables (slow)
- `REBUILD_TABLES = False` -> skip table creation, only fetch + materialize
- After editing `journey_planner_v3.py`, run the reload-code cell above instead of this cell.

In [ ]:
FORCE_PREPARE = False
REBUILD_TABLES = True
REBUILD_PREREQUISITES = True

if planner.prepared and not FORCE_PREPARE:
    print("planner already prepared; skipping prepare(). Set FORCE_PREPARE=True to run it again.")
    prepare_result = {"skipped": True}
else:
    planner = reload_planner_code(preserve_prepared=False)
    prepare_result = bench_prepare(
        planner,
        regions=None,
        rebuild=REBUILD_TABLES,
        rebuild_prerequisites=REBUILD_PREREQUISITES,
    )

prepare_result

"""
On Lausanne:
    v1 40 sec
    v2 40 sec -> cache everything

On whole switzerland
   ~8min
"""

## Benchmark — plan_candidates

> Note: check bellow for a more comprehensive benchmarking

In [16]:

"""
On Max routes = 5
iterations = 100

Average:

On Lausanne:
    v1 111ms
    v2 136.56 ms -> cache everything
    v3 - 11ms        - with stopping creterion
    v4 - 11ms            - with start criterion + stop criterion
    
    v7 - 0.10 ms -> with cacheing
    

On whole switzerland
   v1 avg 3049.15 ms 
   v2 avg 2683 ms -> cache everything
   v3 - 395.51ms        - with stopping creterion | CRAZY DIFFERENCE
   v4 - 393.03 ms           - with start criterion + stop criterion
    
"""

iterations = 100   # ← change this

plan_result = bench_plan_candidates(
    planner,
    iterations=iterations,
    start_stop_id=START_STOP,
    end_stop_id=END_STOP,
    
    
    travel_date="2026-05-11",
    arrival_deadline="09:00",
    max_routes=5,
)
plan_result



  bench_plan_candidates  (n=100)
  run  1/100  

  plan_candidates     total=461.7ms  backward_calls=5  avg_backward=91.9ms  routes=5
  run  2/100    run  3/100    run  4/100    run  5/100    run  6/100    run  7/100    run  8/100    run  9/100    run 10/100    run 11/100    run 12/100    run 13/100    run 14/100    run 15/100    run 16/100    run 17/100    run 18/100    run 19/100    run 20/100    run 21/100    run 22/100    run 23/100    run 24/100    run 25/100    run 26/100    run 27/100    run 28/100    run 29/100    run 30/100    run 31/100    run 32/100    run 33/100    run 34/100    run 35/100    run 36/100    run 37/100    run 38/100    run 39/100    run 40/100    run 41/100    run 42/100    run 43/100    run 44/100    run 45/100    run 46/100    run 47/100    run 48/100    run 49/100    run 50/100    run 51/100    run 52/100    run 53/100    run 54/100    run 55/100    run 56/100    run 57/100    run 58/100    run 59/100    run 60/100    run 61/100    run 62/100    run 63/100    run 64/100    run 65/100    run 66/100    run

{'times_ms': [461.86615293845534,
  0.017171958461403847,
  0.015835976228117943,
  0.01940084621310234,
  0.01158192753791809,
  0.009215902537107468,
  0.008390052244067192,
  0.01209089532494545,
  0.015400117263197899,
  0.01034722663462162,
  0.009085051715373993,
  0.00847293995320797,
  0.008357921615242958,
  0.010765856131911278,
  0.01626298762857914,
  0.01613609492778778,
  0.01147296279668808,
  0.012978212907910347,
  0.016317004337906837,
  0.015804078429937363,
  0.01637195236980915,
  0.01302710734307766,
  0.012957025319337845,
  0.015788013115525246,
  0.015605008229613304,
  0.015737023204565048,
  0.01391698606312275,
  0.013523967936635017,
  0.01615402288734913,
  0.016508856788277626,
  0.013930024579167366,
  0.013011042028665543,
  0.016028992831707,
  0.014430144801735878,
  0.01632911153137684,
  0.0160890631377697,
  0.013245036825537682,
  0.013248063623905182,
  0.013532117009162903,
  0.01929188147187233,
  0.016655074432492256,
  0.014063902199268341,
 

## Benchmark random 200

In [17]:
import random
import statistics

random.seed(42)
stop_list = list(planner.stops)

# bias toward stops that actually have many connections (busy stops)
"""
This ensures you're benchmarking realistic journeys between actual hubs — 
Zürich, Bern, Geneva, Basel, Lausanne — which is what matters in production 
and where the optimizations actually show their effect.
"""

from collections import Counter
stop_freq = Counter()
for conn in planner.connections_by_day["monday"]:
    stop_freq[conn[0]] += 1
    stop_freq[conn[1]] += 1

# take top 200 busiest stops only
busy_stops = [s for s, _ in stop_freq.most_common(200)]

TEST_PAIRS = []
random.seed(42)
#while len(TEST_PAIRS) < 100:
while len(TEST_PAIRS) < 30:
    s, e = random.choice(busy_stops), random.choice(busy_stops)
    if s != e:
        TEST_PAIRS.append((s, e))

In [9]:
all_times = []

for i, (START_STOP, END_STOP) in enumerate(TEST_PAIRS):
    print(f"ON run: {i}")
    result = bench_plan_candidates(
        planner,
        iterations=4,
        start_stop_id=START_STOP,
        end_stop_id=END_STOP,
        travel_date="2026-05-11",
        arrival_deadline="09:00",
        max_routes=5
    )
    all_times.append(result["avg_ms"])

all_times.sort()
print(f"  n:    {len(all_times)}")
print(f"  avg:  {statistics.mean(all_times):.1f}ms")
print(f"  p50:  {all_times[len(all_times)//2]:.1f}ms")
print(f"  p95:  {all_times[int(len(all_times)*0.95)]:.1f}ms")
print(f"  max:  {all_times[-1]:.1f}ms")

ON run: 0

  bench_plan_candidates  (n=4)
  run  1/4  

  plan_candidates     total=472.2ms  backward_calls=5  avg_backward=93.8ms  routes=5
  run  2/4    run  3/4    run  4/4  --------------------------------------------
  avg                       118.14 ms
  min                         0.02 ms
  max                       472.49 ms
  stdev                     236.24 ms
--------------------------------------------
  bench_plan_candidates done    118.14 ms

ON run: 1

  bench_plan_candidates  (n=4)
  run  1/4    plan_candidates     total=2875.4ms  backward_calls=4  avg_backward=718.5ms  routes=0
  run  2/4    run  3/4    run  4/4  --------------------------------------------
  avg                       718.95 ms
  min                         0.01 ms
  max                      2875.75 ms
  stdev                    1437.86 ms
--------------------------------------------
  bench_plan_candidates done    718.95 ms

ON run: 2

  bench_plan_candidates  (n=4)
  run  1/4    plan_candidates     total=536.0ms  backward_calls=1  avg_backward=535.7ms  r

**v2. Basic CSA**
  - n:    100
  - avg:  1794.5ms
  - p50:  1758.4ms
  - p95:  4770.5ms
  - max:  6592.7ms

  All: 11 min run


**v3. With stopping criterion**
  - n:    100
  - avg:  710.3ms
  - p50:  578.7ms
  - p95:  1990.8ms
  - max:  2772.3ms

All 4 min

**v4. With start and stopping criterion**

  - n:    100
  - avg:  665.5ms
  - p50:  495.5ms
  - p95:  1996.0ms
  - max:  2768.0ms

All 4 min

_v5 and forth have all the opt in others_

**v5. Pruning reachable trips with an initial linear scan**

  - n:    100
  - avg:  968.4ms
  - p50:  800.9ms
  - p95:  2214.1ms
  - max:  2849.3ms

**v6. All on v4 and reusing everything from plan_candidates, code optimization**

  - n:    100
  - avg:  470.9ms
  - p50:  487.1ms
  - p95:  1035.3ms
  - max:  1077.7ms

// Comment : v6 tested on another day same as v7

**v6. Again**
  - n:    100
  - avg:  498.4ms
  - p50:  512.4ms
  - p95:  1099.8ms
  - max:  1198.2ms

**v7. v6 and further opt**
  - n:    100
  - avg:  481.3ms
  - p50:  498.9ms
  - p95:  1067.1ms
  - max:  1128.9ms

**v8. With cache**
  - n:    100
  - avg:  483.0ms
  - p50:  504.7ms
  - p95:  1061.1ms
  - max:  1117.3ms

  **v9. Fixed bug and cache**
   - n:    100
   - avg:  153.7ms
   - p50:  127.2ms
   - p95:  265.9ms
   - max:  280.2ms

  **v10. Fixed the bug where 1 route only**

  - n:    100
  - avg:  191.9ms
  - p50:  127.3ms
  - p95:  566.2ms
  - max:  3338.7ms

**v10 fied correctness**

  - n:    30
  - avg:  3241.8ms
  - p50:  2412.8ms
  - p95:  6574.7ms
  - max:  7432.1ms

In [ ]:
# Quick reload for edits in journey_planner_v3.py or tests/test_CSA.py.
# Prepared planner data stays resident in this kernel.
planner = reload_planner_code()

1. Limited walking (Section 3.1) — highest yield remaining
Skip _apply_backward_footpaths entirely if departures[dep_stop] is already set better than what the footpath would produce. Right now you call it for every reachable connection. The paper reports 1.5-2x from this alone, and it's one line of guard logic.
2. Scanning only reachable trips (Section 4.3) — medium yield, medium complexity
Run a cheap forward scan first from start_id to mark which trip_idx values are reachable at all. Then in the backward loop, skip connections whose trip is not reachable. Dramatically cuts connections processed on sparse queries.
3. Reuse trip_reachable across plan_candidates iterations — not in paper but implied
Right now you reinitialize trip_reachable = bytearray(self.n_trips) on every call to route(). Since plan_candidates calls it up to 5 times with the same deadline, trips reachable in iteration 1 are still reachable in iteration 2. You could warm-start it.
4. CSAccel multilevel overlay (Section 5) — highest ceiling, highest complexity
Partition stops into cells, precompute transit connections per cell, only scan relevant subset per query. Paper gets ~10x over base CSA on Germany. But requires heavy preprocessing and weeks of engineering.

In [27]:
all_times = []

for i, (START_STOP, END_STOP) in enumerate(TEST_PAIRS):
    print(f"ON run: {i}")
    result = bench_plan_candidates(
        planner,
        iterations=1,
        start_stop_id=START_STOP,
        end_stop_id=END_STOP,
        travel_date="2026-05-11",
        arrival_deadline="09:00",
        max_routes=5
    )
    all_times.append(result["avg_ms"])

all_times.sort()
print(f"  n:    {len(all_times)}")
print(f"  avg:  {statistics.mean(all_times):.1f}ms")
print(f"  p50:  {all_times[len(all_times)//2]:.1f}ms")
print(f"  p95:  {all_times[int(len(all_times)*0.95)]:.1f}ms")
print(f"  max:  {all_times[-1]:.1f}ms")

ON run: 0

  bench_plan_candidates  (n=1)
  run  1/1  

  plan_candidates     total=397.1ms  backward_calls=5  avg_backward=79.0ms  routes=5
--------------------------------------------
  avg                       397.39 ms
  min                       397.39 ms
  max                       397.39 ms
  stdev                       0.00 ms
--------------------------------------------
  bench_plan_candidates done    397.39 ms

ON run: 1

  bench_plan_candidates  (n=1)
  run  1/1    plan_candidates     total=1949.5ms  backward_calls=4  avg_backward=487.1ms  routes=0
--------------------------------------------
  avg                      1949.77 ms
  min                      1949.77 ms
  max                      1949.77 ms
  stdev                       0.00 ms
--------------------------------------------
  bench_plan_candidates done   1949.77 ms

ON run: 2

  bench_plan_candidates  (n=1)
  run  1/1    plan_candidates     total=535.9ms  backward_calls=1  avg_backward=535.6ms  routes=0
--------------------------------------------
  avg              

In [26]:
if "bench_plan_profile" not in globals():
    import importlib
    import tests.test_CSA as _bench
    importlib.reload(_bench)
    bench_plan_profile = _bench.bench_plan_profile

all_times = []

for i, (START_STOP, END_STOP) in enumerate(TEST_PAIRS):
    print(f"ON run: {i}")
    result = bench_plan_profile(
        planner,
        iterations=1,
        start_stop_id=START_STOP,
        end_stop_id=END_STOP,
        travel_date="2026-05-11",
        arrival_deadline="09:00",
        max_routes=5
    )
    all_times.append(result["avg_ms"])

all_times.sort()
print(f"  n:    {len(all_times)}")
print(f"  avg:  {statistics.mean(all_times):.1f}ms")
print(f"  p50:  {all_times[len(all_times)//2]:.1f}ms")
print(f"  p95:  {all_times[int(len(all_times)*0.95)]:.1f}ms")
print(f"  max:  {all_times[-1]:.1f}ms")

ON run: 0

  bench_plan_profile  (n=1)
  run  1/1    plan_profile        total=3214.1ms  scanned=923337  routes=5
--------------------------------------------
  avg                      3229.16 ms
  min                      3229.16 ms
  max                      3229.16 ms
  stdev                       0.00 ms
--------------------------------------------
  bench_plan_profile done   3229.16 ms

ON run: 1

  bench_plan_profile  (n=1)
  run  1/1    plan_profile        total=2397.3ms  scanned=923337  routes=1
--------------------------------------------
  avg                      2412.78 ms
  min                      2412.78 ms
  max                      2412.78 ms
  stdev                       0.00 ms
--------------------------------------------
  bench_plan_profile done   2412.78 ms

ON run: 2

  bench_plan_profile  (n=1)
  run  1/1    plan_profile        total=2672.7ms  scanned=923337  routes=0
--------------------------------------------
  avg                      2681.14 ms
  min      

# Benchmark new correctness

## Benchmark — route (forward CSA)

Set `end_id=None` for one-to-all.

In [ ]:
iterations = 6   # ← change this

route_result = bench_route(
    planner,
    iterations=iterations,
    start_id=START_STOP,
    end_id=END_STOP,
    departs="08:30",
    day="monday",
)
route_result



## Benchmark — route_backward (single backward scan)

In [ ]:
iterations = 6   # ← change this

backward_result = bench_route_backward(
    planner,
    iterations=iterations,
    start_id=START_STOP,
    end_id=END_STOP,
    deadline="09:00",
    day="monday",
)
backward_result

## Summary table

In [15]:
import pandas as pd

rows = []
for label, r in [
    ("plan_candidates", plan_result),
    ("route (p2p)",     route_result),
    #("route_backward",  backward_result),
]:
    rows.append({
        "benchmark": label,
        "avg_ms":   round(r["avg_ms"],   2),
        "min_ms":   round(r["min_ms"],   2),
        "max_ms":   round(r["max_ms"],   2),
        "stdev_ms": round(r["stdev_ms"], 2),
    })

pd.DataFrame(rows).set_index("benchmark")

,avg_ms,min_ms,max_ms,stdev_ms
benchmark,,,,
plan_candidates,3049.15,3013.15,3070.78,22.23
route (p2p),5.84,2.68,18.59,6.29
